# Scenario 06 — Agent Setup

A master authorizes an external address (an **agent**) to act on their behalf
for task-related operations (register providers to sources, attempt tasks,
submit responses). The Order (`0x306Dc3fF...`) is the built-in scheduling agent;
any address can play this role.

**Role:** Master.

**Prerequisites:**
- `MASTER_PRIVATE_KEY` — funded master wallet on testnet
- Master-provider pairing is **already done** by the Provider App. This notebook
  does not set up the pairing — that is Provider App territory, not a typical
  SDK-user task.

**Flow:**

```
Master
  │
  ├─ 1. terminal.set_agent(agent, True)    → Receipt
  ├─ 2. terminal.is_agent_of(master, agent) → True
  ├─ 3. master.set_agent(agent, True)      → same thing via Master instance
  └─ 4. terminal.revoke_agent(agent)       → convenience wrapper
```

Scenario 07 picks up where this leaves off: the agent uses its own key to drive
a full task lifecycle on the master's behalf.


## 0. Setup


In [ ]:
import os

from eth_account import Account

from ogpu import ChainConfig, ChainId
from ogpu.protocol import Master, terminal

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

MASTER_KEY = os.environ["MASTER_PRIVATE_KEY"]
MASTER_ADDR = Account.from_key(MASTER_KEY).address

# "The Order" — OpenGPU's default scheduling agent
THE_ORDER = "0x306Dc3fF30254675B209D916475094401aCC4a1E"

print(f"Master: {MASTER_ADDR}")
print(f"Agent : {THE_ORDER}  (The Order)")


## 1. Master authorizes the agent

`terminal.set_agent(agent, True, signer=MASTER_KEY)` lets the given address sign
transactions on behalf of the master. The Terminal contract emits an `AgentSet`
event. Only a Client or Master can call this (not a Provider — see PRD N1).


In [ ]:
receipt = terminal.set_agent(THE_ORDER, True, signer=MASTER_KEY)
print(f"set_agent tx: {receipt.tx_hash}")
print(f"Block: {receipt.block_number}")


## 2. Verify the authorization landed


In [ ]:
is_agent = terminal.is_agent_of(MASTER_ADDR, THE_ORDER)
print(f"isAgentOf(master, agent) = {is_agent}")
assert is_agent is True


## 3. Same call via the `Master` instance

`Master.set_agent(...)` is a thin wrapper over `terminal.set_agent`. Useful if
your code already holds a `Master` instance for other reads/writes.


In [ ]:
master = Master(MASTER_ADDR)
# Idempotent — re-setting to True is a no-op on-chain (same tx cost though)
receipt = master.set_agent(THE_ORDER, True, signer=MASTER_KEY)
print(f"Master.set_agent tx: {receipt.tx_hash}")


## 4. Revoke (for cleanup / rotation)

`terminal.revoke_agent(agent, signer=...)` is a one-liner for
`set_agent(agent, False, ...)`. Use it when rotating keys or removing a
compromised agent.

**Skip this cell if you want the agent active for scenario 07.**


In [ ]:
# Uncomment to revoke
# receipt = terminal.revoke_agent(THE_ORDER, signer=MASTER_KEY)
# print(f"revoke tx: {receipt.tx_hash}")
# assert terminal.is_agent_of(MASTER_ADDR, THE_ORDER) is False


## 5. (Bonus) Who else can call set_agent?

Clients can also set agents for themselves — the agent can then publish tasks,
confirm responses, and make any client-side call on behalf of the client. This
is the "client-side agent" pattern, parallel to the master-side pattern shown
above.

```python
from ogpu.protocol import terminal
terminal.set_agent(agent_addr, True, signer=CLIENT_KEY)
```

Providers cannot set agents — they are not principals in the agent-delegation
model (see PRD N1). Provider-related operations are instead covered by the
master who manages the provider.
